# Unit 6 solutions: evaluation deepened and the first blend

These reference solutions close Part 1. We have five retrieval paths and a scoreboard, and here we answer *which* recommender is best, and best at *what*. First two **ranking metrics** sharpen the question: precision@k (the fraction of the top-k slots that are relevant) and NDCG@k (which additionally rewards putting the right book *higher*), each computed by hand and then confirmed against the shipped `bookrec` functions to floating-point precision. Then three **beyond-accuracy** metrics -- catalog coverage, intra-list diversity, novelty -- show that two paths with similar hit@k can serve wildly different shelves. Finally we build the book's **first blended recommender**: calibrate each path to a common scale, take a weighted union with `bookrec.blend`, order it with `bookrec.rank`, and read the blended scoreboard *honestly*. A leave-one-path-out ablation delivers the headline surprise -- dropping popularity *raises* hit@10 -- and the two Challenge exercises sweep the retrieval pool and reproduce the calibration pitfall behind it.

Everything is measured on the **validation** split (`val`), with each reader's **seen** set their train-positive history; the `test` holdout stays sealed (only Checkpoint A unseals it). `val` is the split we *select* on -- that is its job -- but the discipline is not to report the single best number from a `val` sweep as expected performance (selecting *and* reporting on the same split biases the estimate upward); the blend **config is PINNED** (per-path pool 30; weights {popularity 0.25, lexical 0.5, item-item 1.0, mf 1.0}), chosen in advance rather than searched, which sidesteps the sweep. Every toy number is seeded with `SEED = 0`, so the notebook runs top-to-bottom deterministically, and each exercise ends with an `assert` on a computed result so a wrong number fails loudly rather than printing quietly. The honest thesis, stated up front and never overclaimed below: on this data at this pinned config the blend beats the best single path (MF) **and** covers about 4.5x more of the catalog -- but it is *not* a law that "a blend always wins", and the ablation shows "every path contributes" is outright false here.

In [ ]:
import json
import math
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    blend,
    catalog_coverage,
    hit_rate_at_k,
    intra_list_diversity,
    ndcg_at_k,
    novelty,
    precision_at_k,
    rank,
    run_blended_scoreboard,
    run_validation_scoreboard,
)

SEED = 0
K = 10        # we report every metric at the top-10 cutoff
POOL = 30     # the PINNED per-path retrieval pool (chosen in advance, not swept on val)
# The PINNED blend weights, measured on this data (reader-dependent paths earn the big weights;
# popularity is kept small only for cold-reader robustness, not for accuracy or coverage -- the
# ablation in Exercise 5 shows its marginal accuracy is about zero).
WEIGHTS = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}

# --- the real interaction log (the same substrate as Units 1-5) ----------------------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)                      # 2000 books -- the catalog coverage denominator
interactions_path = data_dir / "interactions.csv.gz"
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])           # 60 readers the personalized paths cannot serve

# The interaction log as plain row dicts -- the row-mapping log the paths duck-type.
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")

# Each reader's train-positive history is their `seen` (exclusion) set. We grade on `val`; the
# `test` split stays sealed -- no cell here may touch it.
seen_by = defaultdict(set)
for row in rows:
    if row["split"] == "train" and row["label"] == 1:
        seen_by[row["reader_id"]].add(row["item_id"])

print(f"catalog books: {len(catalog_ids)}   cold readers: {len(cold_readers)}")

## Exercise 1 - Precision@k and NDCG@k, by hand then via `bookrec`

Hit@k only asks *did any relevant book make the top k?* -- it saturates at the first hit and sees nothing else. Two richer **ranking metrics** look deeper. **Precision@k** is the fraction of the top-k slots that are relevant, `|top_k AND relevant| / k` (the denominator is always k). **NDCG@k** goes further and rewards putting a relevant book *higher*: a relevant book at 1-based rank r contributes a discounted gain `1 / log2(r + 1)`, and the sum (the DCG) is normalized by the ideal DCG (IDCG) -- the DCG of the best possible ordering, which front-loads every relevant book -- so the score lands in [0, 1].

Below is a tiny ranked recommendation list and the reader's held-out relevant books. Compute hit@5, precision@5, and NDCG@5 **by hand** (use `math.log(r + 1, 2)` for the log base 2), then call the shipped `precision_at_k` and `ndcg_at_k` and confirm they match your hand numbers to floating-point precision. In two or three sentences, say what precision@k rewards that hit@k ignores, and what NDCG@k rewards that *both* of the others ignore.


In [ ]:
recommendations = [11, 4, 7, 2, 9]   # five books in ranked order (rank 1 first)
relevant = {7, 2}                     # the books this reader actually engaged in val

hit_by_hand = 1.0                     # some top-5 book (7 and 2) is relevant -> a hit
precision_by_hand = 2 / 5             # 2 of the 5 slots (books 7 and 2) are relevant

# DCG: sum 1/log2(r+1) over the 1-based ranks r whose book is relevant. The relevant books sit at
# ranks 3 (book 7) and 4 (book 2).
dcg_by_hand = 1 / math.log(3 + 1, 2) + 1 / math.log(4 + 1, 2)
# IDCG: the ideal ordering puts both relevant books first, at ranks 1 and 2.
idcg_by_hand = 1 / math.log(1 + 1, 2) + 1 / math.log(2 + 1, 2)
ndcg_by_hand = dcg_by_hand / idcg_by_hand

# The shipped metrics -- confirm they agree with the hand computation.
precision_lib = precision_at_k(recommendations, relevant, 5)
ndcg_lib = ndcg_at_k(recommendations, relevant, 5)
metrics_match = bool(
    np.isclose(precision_by_hand, precision_lib) and np.isclose(ndcg_by_hand, ndcg_lib)
)
metrics_explanation = (
    "Hit@5 is 1.0 the instant any relevant book lands in the top 5 and then sees nothing more, so "
    "it cannot tell a list with one relevant book from a list with two. Precision@5 counts the "
    "relevant SLOTS (2/5 = 0.4 here), rewarding packing more of the top k with relevant books. "
    "NDCG@5 goes one step further and is the only one of the three that rewards WHERE a relevant "
    "book sits -- a relevant book higher up earns a larger discounted gain, so order itself scores."
)

print(f"hit@5 by hand       : {hit_by_hand}")
print(f"precision@5 by hand : {precision_by_hand}   lib: {precision_lib}")
print(f"ndcg@5 by hand      : {ndcg_by_hand:.6f}   lib: {ndcg_lib:.6f}")
print(f"hand numbers match the library: {metrics_match}")

assert precision_lib == precision_by_hand          # 2/5 exactly
assert np.isclose(ndcg_lib, ndcg_by_hand)          # the shipped NDCG matches the hand DCG/IDCG
assert metrics_match, "hand-computed precision@5 and NDCG@5 must match the shipped functions"


## Exercise 2 - Placement matters: why a ranking metric, not RMSE

A recommender returns an *ordered* list, and a reader scans it from the top -- so *where* a relevant book sits should count. Here are two rankings of the **same** relevant set: one puts both relevant books at the very top, the other buries them at the bottom. Compute hit@5, precision@5, and NDCG@5 for each.

You will find hit@5 and precision@5 are **identical** for the two rankings (both lists contain the same two relevant books somewhere in the top 5), while NDCG@5 is much higher for the top-loaded ranking -- NDCG is the only one of the three that *sees the order*. In two or three sentences, explain why this makes a ranking metric (not rating-prediction RMSE) the right lens for an implicit-feedback top-k problem: we have no held-out *ratings* to take a squared error against, and the reader only ever sees a ranked shortlist.


In [ ]:
relevant = {7, 2}
ranking_high = [7, 2, 11, 4, 9]   # the two relevant books at the very top
ranking_low = [11, 4, 9, 7, 2]    # the SAME two relevant books, buried at the bottom

hit_high = hit_rate_at_k(ranking_high, relevant, 5)
hit_low = hit_rate_at_k(ranking_low, relevant, 5)
precision_high = precision_at_k(ranking_high, relevant, 5)
precision_low = precision_at_k(ranking_low, relevant, 5)
ndcg_high = ndcg_at_k(ranking_high, relevant, 5)
ndcg_low = ndcg_at_k(ranking_low, relevant, 5)

hit_is_order_blind = bool(hit_high == hit_low)
precision_is_order_blind = bool(precision_high == precision_low)
ndcg_sees_order = bool(ndcg_high > ndcg_low)
ranking_explanation = (
    "Both lists hold the same two relevant books somewhere in the top 5, so hit@5 (1.0 vs 1.0) and "
    "precision@5 (0.4 vs 0.4) are identical -- neither looks at placement. NDCG@5 is much higher "
    "for the top-loaded list (1.0 vs ~0.50) because it discounts gain by rank, so it is the only "
    "metric that rewards surfacing a relevant book early. A ranking metric, not RMSE, is the right "
    "lens here: implicit feedback gives us engaged/not-engaged, not held-out star ratings to take a "
    "squared error against, and the reader only ever scans a ranked shortlist -- so the question is "
    "always 'is the right book near the top?', which is exactly what NDCG measures."
)

print(f"             hit@5   precision@5   ndcg@5")
print(f"top-loaded : {hit_high:.3f}   {precision_high:.3f}         {ndcg_high:.4f}")
print(f"bottom     : {hit_low:.3f}   {precision_low:.3f}         {ndcg_low:.4f}")

# hit@k and precision@k are order-blind here; only NDCG separates the two rankings.
assert hit_is_order_blind and precision_is_order_blind
assert ndcg_sees_order, "NDCG must reward the top-loaded ranking over the buried one"
assert ndcg_high == 1.0  # both relevant books at ranks 1-2 is the ideal ordering


## Exercise 3 - Beyond accuracy: coverage, diversity, novelty

A recommender is more than hit@k. Three **beyond-accuracy** metrics show it. **Catalog coverage** is the fraction of the 2000-book catalog a path ever recommends to *anyone* -- how much of the shelf it can reach. **Intra-list diversity** of one reader's top-k is `1 - mean pairwise similarity` -- high when the list spans item space rather than ten near-duplicates. **Novelty** is the mean self-information `-log2 p_hat` of the recommended books (`p_hat` = a book's share of the train positives), so rarer books are more novel.

Fit three Part-1 paths (popularity, item-item CF, and matrix factorization -- **the MF fit is the only slow one, ~15 s; keep this cell at four fits or fewer**; later exercises REUSE these fitted objects). Measure MF's and item-item's catalog coverage with a single-path blended scoreboard, then for one warm reader compute the intra-list diversity of the item-item top-10 (using the fitted item-item cosine as the book-book similarity) and the novelty of both the item-item and the popularity top-10. In two or three sentences, contrast the paths: MF scores highest on hit@k (Unit 5) yet reaches only a sliver of the catalog, while item-item spreads far wider; popularity's picks are the *least* novel.


In [ ]:
# Three fits in this cell (the MF fit is ~15 s). Every later exercise reuses these fitted paths.
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
item_item = bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
mf = bookrec.MatrixFactorizationPath(seed=SEED).fit(rows, catalog=catalog_ids)


def coverage_of(name, path):
    # A single-path blend is just that path; the blended scoreboard reports its catalog coverage.
    return run_blended_scoreboard(
        {name: path}, interactions_path, catalog_ids=catalog_ids,
        k=K, pool=POOL, weights={name: 1.0}, cold_readers=cold_readers,
    ).catalog_coverage


mf_coverage = coverage_of("mf", mf)                 # narrow (~0.07)
item_item_coverage = coverage_of("item-item", item_item)  # broad (~0.49)

# One warm reader (never a cold reader), their seen history, and the fitted item-item similarity.
reader_id = sorted(reader for reader in seen_by if reader not in cold_readers)[0]
seen = seen_by[reader_id]
similarity = item_item.similarity                                  # 2000 x 2000 cosine matrix
sim_col = {item_id: j for j, item_id in enumerate(item_item.artifact()["item_ids"])}

item_item_recs = rank(item_item.retrieve(reader_id, {"seen": seen}, POOL), n=K, exclude=seen)
popularity_recs = rank(popularity.retrieve(reader_id, {"seen": seen}, POOL), n=K, exclude=seen)
item_item_ids = [c.item_id for c in item_item_recs]
popularity_ids = [c.item_id for c in popularity_recs]

diversity_item_item = intra_list_diversity(
    item_item_ids, lambda a, b: similarity[sim_col[a], sim_col[b]]
)
novelty_item_item = novelty(item_item_ids, popularity.counts)
novelty_popularity = novelty(popularity_ids, popularity.counts)   # LOWER (common books)
popularity_least_novel = bool(novelty_popularity < novelty_item_item)
beyond_accuracy_explanation = (
    "Hit@k is not the whole story. MF tops the hit@k board (Unit 5) yet reaches only ~7% of the "
    "2000-book catalog, while item-item CF spreads across ~49% of the shelf -- about 6.6x wider "
    "reach for a path that scores only a little lower. Popularity, meanwhile, recommends the "
    "best-known books to everyone, so its picks carry the LEAST novelty (lowest mean "
    "self-information): it serves crowd-pleasers, not discoveries."
)

print(f"catalog coverage  mf: {mf_coverage:.4f}   item-item: {item_item_coverage:.4f}")
print(f"warm reader       id: {reader_id}")
print(f"intra-list diversity (item-item top-10): {diversity_item_item:.4f}")
print(f"novelty  item-item: {novelty_item_item:.4f}   popularity: {novelty_popularity:.4f}")
print(f"popularity is the least novel: {popularity_least_novel}")

# item-item reaches far more of the catalog than MF, despite MF's higher hit@k.
assert item_item_coverage > 5 * mf_coverage, "item-item must cover far more catalog than MF"
assert 0.0 < diversity_item_item <= 1.0
assert popularity_least_novel, "popularity's crowd-pleasers must be less novel than item-item's picks"


## Exercise 4 - The first blend: calibrate, weight, union -- read it honestly

Now combine the paths. `bookrec.blend` takes `{path_name: [candidate, ...]}`, **calibrates** each path to a common [0, 1] scale (so a path with big raw scores cannot dominate), sums the weighted calibrated scores per book, and records which paths proposed each pick (its provenance); `bookrec.rank(exclude=seen)` then drops seen books and keeps the top k. First watch the mechanism for one reader, then measure the whole blend on `val` at the **PINNED** config (pool 30, the pinned `WEIGHTS`).

Fit the one remaining path (lexical), reuse the Exercise-3 fits, and compute the blended scoreboard's hit@10 and catalog coverage. Compare them against MF, the best single path (hit ~0.276, coverage ~0.0745). You should find the blend both **beats the best single path on hit@10** (~0.306) and **covers far more catalog** (~0.333, about 4.5x MF). Then write two or three **honest** sentences: this is a win *on this data at this pinned config*, not a law that "a blend always wins"; the pool and weights are pinned in advance rather than searched, so you never report the single best number from a `val` sweep as the blend's expected performance.

In [ ]:
lexical = bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
paths = {"popularity": popularity, "lexical": lexical, "item-item": item_item, "mf": mf}

# See the mechanism for one reader: retrieve a POOL from each path, then calibrate + weight + union.
per_path = {name: path.retrieve(reader_id, {"seen": seen}, POOL) for name, path in paths.items()}
blended_candidates = blend(per_path, weights=WEIGHTS)
top_blend = rank(blended_candidates, n=K, exclude=seen)
top_blend_provenance = [c.provenance for c in top_blend]   # which paths backed each pick


def blended_hit_cov(path_map, weight_map):
    result = run_blended_scoreboard(
        path_map, interactions_path, catalog_ids=catalog_ids,
        k=K, pool=POOL, weights=weight_map, cold_readers=cold_readers,
    )
    return result.hit_rate_at_k, result.catalog_coverage


blend_hit, blend_cov = blended_hit_cov(paths, WEIGHTS)             # ~0.306 / ~0.333
mf_hit, mf_cov = blended_hit_cov({"mf": mf}, {"mf": 1.0})          # ~0.276 / ~0.0745
blend_beats_best_single = bool(blend_hit >= mf_hit)
blend_covers_more = bool(blend_cov > mf_cov)
blend_reading = (
    "At this PINNED config the blend wins on both axes: hit@10 ~0.306 edges past MF's 0.276 (the "
    "best single path), and catalog coverage ~0.333 is about 4.5x MF's 0.0745 -- so the blend not "
    "only ranks a little better but reaches far more of the shelf. This is a result ON THIS DATA "
    "AT THIS CONFIG, not a universal law that 'a blend always wins'; a different dataset, pool, or "
    "weight vector could easily hand the crown back to a single path. `val` is the split we select "
    "on, but the pool and weights are PINNED in advance rather than searched -- so this number is "
    "not the single best result of a val sweep reported as expected performance, which would bias "
    "it upward; test stays sealed for Checkpoint A's one final read."
)

print(f"top blend provenance (first 3): {top_blend_provenance[:3]}")
print(f"blend  hit@10 {blend_hit:.3f}   coverage {blend_cov:.4f}")
print(f"MF     hit@10 {mf_hit:.3f}   coverage {mf_cov:.4f}")
print(f"coverage ratio blend / MF: {blend_cov / mf_cov:.2f}x")
print(f"blend beats best single: {blend_beats_best_single}   covers more: {blend_covers_more}")

# The pinned-config blend clears the best single path on hit@10 and covers >= 3x its catalog.
assert blend_hit >= mf_hit, "pinned blend hit@10 must be at least the best single path (~0.306 vs 0.276)"
assert blend_cov >= 3 * mf_cov, "pinned blend must cover at least 3x MF's catalog (~0.333 vs 0.0745)"
assert blend_beats_best_single and blend_covers_more

## Exercise 5 - Leave-one-path-out: the ablation that surprises

Does *every* path earn its place? Run a **leave-one-path-out** ablation: blend the other three paths (at their pinned weights) and read hit@10, dropping each path in turn. Reuse the Exercise-3/4 fits -- **no refitting**.

The headline result is counter-intuitive: dropping **popularity** *raises* hit@10 (from ~0.306 to ~0.308), while dropping any **reader-dependent** path (lexical, item-item, or MF) *lowers* it. The mechanism is the calibration you saw in Exercise 4: min-max maps every path's top candidate to score 1.0, so a reader-*independent* popularity path ties its handful of head books with every reader's true top pick -- crowding the top of the pool without being right for anyone in particular. In three or four sentences, explain why this means "every path contributes" is **false** here, what actually establishes a path's contribution (its *measured* leave-one-out change -- reader-dependence explains why popularity is weak but is not by itself enough to guarantee positive value), and why the pinned weights still keep popularity at a small 0.25 (cold-reader robustness, not its hit@10 or its coverage).

In [ ]:
# Leave-one-path-out at the pinned weights (reuse the fitted paths; no refitting).
ablation_hit = {}
for name in paths:
    sub = {n: p for n, p in paths.items() if n != name}
    sub_weights = {n: WEIGHTS[n] for n in sub}
    ablation_hit[name] = blended_hit_cov(sub, sub_weights)[0]

drop_popularity_hit = ablation_hit["popularity"]   # ~0.308
drop_mf_hit = ablation_hit["mf"]                   # ~0.260
dropping_popularity_raises_hit = bool(drop_popularity_hit > blend_hit)   # 0.308 > 0.306
dropping_mf_lowers_hit = bool(drop_mf_hit < blend_hit)
ablation_explanation = (
    "Dropping the reader-INDEPENDENT popularity path RAISES hit@10 (0.308 > 0.306), while dropping "
    "any reader-DEPENDENT path -- lexical, item-item, or MF -- LOWERS it. So 'every path "
    "contributes' is simply false here: what settles a path's value is its MEASURED leave-one-out "
    "change, not a story about its design -- reader-dependence explains WHY popularity is weak "
    "here, but it is the measured number that confirms its marginal value is about zero. The "
    "mechanism is calibration: min-max sends every path's top candidate to score 1.0, so "
    "popularity's handful of best-known books tie the very top of the pool for everyone, displacing "
    "a reader's true top pick without being personalized. We keep popularity at a small 0.25 rather "
    "than dropping it entirely for ONE reason -- cold-reader robustness: it is the only path that "
    "can serve the 60 cold readers the others return [] for. It is not kept for accuracy (its "
    "marginal value is about zero) nor for coverage (dropping it actually raises coverage here)."
)

print("leave-one-path-out hit@10 (full blend =", f"{blend_hit:.3f}):")
for name, hit in ablation_hit.items():
    arrow = "UP  " if hit > blend_hit else "down"
    print(f"  drop {name:10s} -> {hit:.3f}  ({arrow})")
print(f"dropping popularity raises hit@10: {dropping_popularity_raises_hit}")
print(f"dropping MF lowers hit@10        : {dropping_mf_lowers_hit}")

# The surprise: removing the reader-independent popularity path does NOT lower hit@10 (it raises it),
# while removing a reader-dependent path does.
assert dropping_popularity_raises_hit, "dropping popularity must not lower hit@10 (it raises it here)"
assert dropping_mf_lowers_hit, "dropping the reader-dependent MF path must lower hit@10"
assert drop_popularity_hit >= blend_hit, "the ablation refutes 'every path contributes'"

## Exercise 6 - Challenge: sweep the per-path pool size

The per-path retrieval pool is the number of candidates each path contributes *before* the blend. Pinned at 30, but it matters. Sweep it over a **small grid** -- keep this cell to at most four blended-scoreboard passes (no refits; the paths are already fitted) -- at the pinned `WEIGHTS`, and record hit@10 for each pool size.

You should see hit@10 rise as the pool grows: too small a pool (near k) starves the blend, because each path can only offer a few candidates and the union barely differs from a single path. hit@10 keeps nudging up past pool 30 here -- but a bigger pool is not free: catalog coverage *falls* as the pool grows (about 0.379 at pool 10, 0.333 at 30, 0.3075 at 50), and each extra candidate costs retrieval budget. We **pin** 30 to balance hit against coverage and budget, rather than report the single pool with the best `val` hit@10 as expected performance (selecting *and* reporting on the same split would bias that number upward). In one or two sentences, explain why a pool near k is too small and give one honest reason to pin 30 rather than pick the val-max pool.

In [ ]:
# At most four blended-scoreboard passes in this cell; NO refits (reuse the fitted `paths`).
pool_grid = [10, 30, 50]
hit_by_pool = {}
for pool in pool_grid:
    hit_by_pool[pool] = run_blended_scoreboard(
        paths, interactions_path, catalog_ids=catalog_ids, k=K, pool=pool,
        weights=WEIGHTS, cold_readers=cold_readers,
    ).hit_rate_at_k

bigger_pool_helps = bool(hit_by_pool[30] > hit_by_pool[10])   # ~0.306 > ~0.278
pool_explanation = (
    "A pool near k (10) starves the blend: each path can offer only a handful of candidates, so the "
    "weighted union barely differs from a single path and the paths cannot reinforce each other -- "
    "hit@10 is only ~0.278. hit@10 keeps nudging up past pool 30 (~0.316 at 50), but a bigger pool "
    "is not free: catalog coverage FALLS as the pool grows (about 0.379 at pool 10, 0.333 at 30, "
    "0.3075 at 50) and each extra candidate costs retrieval budget. We PIN 30 to balance hit "
    "against coverage and budget, rather than report the single pool with the best val hit@10 as "
    "the blend's expected performance -- selecting AND reporting on the same split would bias that "
    "number upward."
)

print("hit@10 by per-path pool size (pinned weights):")
for pool in pool_grid:
    print(f"  pool {pool:3d} -> {hit_by_pool[pool]:.3f}")
print(f"pool 30 beats pool 10: {bigger_pool_helps}")

# A bigger pool helps (30 > 10), and hit keeps rising past the pinned 30 -- which is exactly why
# pinning (rather than picking the val-max pool) matters.
assert bigger_pool_helps, "a pool of 30 must beat a starved pool of 10"
assert hit_by_pool[50] > hit_by_pool[30], "hit@10 keeps rising past the pinned pool 30 on val"

## Exercise 7 - Challenge: reproduce the calibration pitfall

This is the mechanism behind Exercise 5's surprise, shown directly. `bookrec.blend` **min-max calibrates** each path on its own, so every path's *top* candidate gets score 1.0 -- no matter how good that path is. A reader-independent, flat path (popularity) therefore lands its single most-popular book at the very top of the pool, tying the top of a strong, reader-dependent path (item-item).

For one warm reader, retrieve a pool from item-item and from popularity. Blend each **alone** (equal weight) and read the top calibrated score of each -- both should be exactly 1.0. Then blend the two together at equal weight and inspect the provenance of the top few pooled picks: popularity's head book sits level with the reader's personalized item-item pick. In three or four sentences, explain how min-max calibration lets a weak/flat path *tie* a strong one at the top of the pool, connect it to "dropping popularity raises hit@10", and name one fix a later unit could use (a learned reranker instead of raw min-max plus fixed weights).


In [ ]:
# One warm reader (reuse Exercise 3's reader_id / seen) and a pool from each of two paths.
item_item_pool = item_item.retrieve(reader_id, {"seen": seen}, POOL)
popularity_pool = popularity.retrieve(reader_id, {"seen": seen}, POOL)

# Blend each path ALONE (default equal weight) -- min-max puts each path's own top candidate at 1.0.
item_item_only = rank(blend({"item-item": item_item_pool}), n=1, exclude=seen)
popularity_only = rank(blend({"popularity": popularity_pool}), n=1, exclude=seen)
item_item_top_score = item_item_only[0].score    # 1.0 after min-max calibration
popularity_top_score = popularity_only[0].score  # ALSO 1.0 (the pitfall)
top_scores_tie = bool(np.isclose(item_item_top_score, popularity_top_score))

# Blend the two together at equal weight; read the provenance of the top pooled picks.
pitfall_blend = rank(
    blend({"item-item": item_item_pool, "popularity": popularity_pool}), n=5, exclude=seen
)
pitfall_top_provenance = [c.provenance for c in pitfall_blend]   # popularity ties the top
pitfall_explanation = (
    "Min-max calibration rescales each path's pool on its own, so EVERY path's top candidate -- the "
    "best item-item pick AND popularity's single most-read book -- lands at score 1.0, regardless "
    "of how good the path actually is. Blended together at equal weight, popularity's flat, "
    "reader-independent head book therefore ties the reader's personalized item-item pick at the "
    "very top of the pool (see the mixed provenance of the top picks). That is exactly why dropping "
    "popularity RAISES hit@10: its calibrated-to-1.0 head books crowd out a reader's true top pick "
    "without ever being right for that reader. A later unit's fix is a LEARNED reranker that scores "
    "the pooled candidates jointly, instead of trusting raw min-max calibration plus fixed weights."
)

print(f"item-item top calibrated score: {item_item_top_score}")
print(f"popularity top calibrated score: {popularity_top_score}")
print(f"the two paths' top scores tie at 1.0: {top_scores_tie}")
print(f"top-5 pooled provenance: {pitfall_top_provenance}")

# The pitfall: min-max hands BOTH paths' top candidate a score of exactly 1.0, so a flat path ties
# a strong one at the top of the pool.
assert item_item_top_score == 1.0 and popularity_top_score == 1.0
assert top_scores_tie, "min-max calibration ties every path's top candidate at 1.0 -- the pitfall"
assert "popularity" in pitfall_top_provenance, "popularity's head book must reach the top of the pool"
